Author: Amanda Baright
Purpose: ST 554 Project 2
Date: 03.25.2026

# ST 554 Project 2

## Part I: Using my_class.py

This section of the proect will focus on using an air quality dataset to test my created python class called `SparkDataCheck` in a python file called`my_class.py`. This python class will work on a Spark SQL style data frame and has a few methods. There are two `@classmethods` that create instances from reading in csv files and reading in pandas dataframes. There are also a few validation methods that append Boolean columns to the dataframe and serve to check if the numeric columns fall under a user specified range (`check_numeric_range`), if string columns fall within a user specified set of levels (`check_string_levels`), and if there are missing values in a column (`check_missing`). The class then wraps up with two summarization methods that report the min and max of numeric columns (`report_min_max`) and reports the counts associated with one or two string columns (`report_counts`). Both summarization methods account for a user provided grouping variable and return a pandas dataframe of the summarization.

The following section will focus on using this air quality dataset (https://www4.stat.ncsu.edu/online/datasets/air.csv) to test the `SparkDataCheck` class and its methods. We first need to import our packages and create a Spark Session.

In [36]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession
import numpy as np
import pandas as pd
import pyspark.pandas as ps

spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

We then need to import the `my_class` script so that it can be access in this `.ipynb` file.

In [37]:
from my_class import SparkDataCheck

Now that we imported in `SparkDataCheck`, we want to read in the air quality data as a csv file. For this, the csv file will need to be downloaded and uploaded into the JupyterHub. We will then use the class methods to create an instance of the class from the csv file.

In [38]:
air_quality = SparkDataCheck.from_csv(spark, 'air.csv')

print("Data successfully loaded. Schema: ")
air_quality.df.printSchema()

Data successfully loaded. Schema: 
root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



For ease of use, we want to do some basic data cleaning before our examples to rename our columns, convert the `-200` values to `Null` and we can create a binary variable with low vs high temperature using the median as the middle point to determine the binary category.

In [39]:
# Remap the column names
rename_map = {
    "_c0": "id",
    "CO(GT)": "co_gt",
    "PT08.S1(CO)": "sensor_co",
    "NMHC(GT)": "nmhc_gt",
    "C6H6(GT)": "benzene_gt",
    "PT08.S2(NMHC)": "sensor_nmhc",
    "NOx(GT)": "nox_gt",
    "PT08.S3(NOx)": "sensor_nox",
    "NO2(GT)": "no2_gt",
    "PT08.S4(NO2)": "sensor_no2",
    "PT08.S5(O3)": "sensor_o3",
    "T": "temp",
    "RH": "rel_humidity",
    "AH": "abs_humidity"
}

# Now apply the new names to the old names, and also replace -200 with None
for old, new in rename_map.items():
    if old in air_quality.df.columns:
        air_quality.df = air_quality.df.withColumn(
            new, 
            F.when(F.col(f"`{old}`") == -200, None).otherwise(F.col(f"`{old}`"))
        )
        if old != new:
            air_quality.df = air_quality.df.drop(old)
            
# Now print and check the new schema
print("Cleaned Schema:")
air_quality.df.printSchema()

Cleaned Schema:
root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- id: integer (nullable = true)
 |-- co_gt: double (nullable = true)
 |-- sensor_co: integer (nullable = true)
 |-- nmhc_gt: integer (nullable = true)
 |-- benzene_gt: double (nullable = true)
 |-- sensor_nmhc: integer (nullable = true)
 |-- nox_gt: integer (nullable = true)
 |-- sensor_nox: integer (nullable = true)
 |-- no2_gt: integer (nullable = true)
 |-- sensor_no2: integer (nullable = true)
 |-- sensor_o3: integer (nullable = true)
 |-- temp: double (nullable = true)
 |-- rel_humidity: double (nullable = true)
 |-- abs_humidity: double (nullable = true)



We now want to create a binary variable we can use to test the string method. Here we will find the median temperature and create a binary low vs high category to determine if the given temperature is lower or higher than the median value.

In [40]:
median_temp = air_quality.df.approxQuantile("temp", [0.5], 0.01)[0]
print(f"The median temperature is: {median_temp}")

# Create the binary variable
air_quality.df = air_quality.df.withColumn(
    "temp_category",
    F.when(F.col("temp").isNull(), None)
     .when(F.col("temp") <= median_temp, "Low")
     .otherwise("High")
)

The median temperature is: 17.7


In case we need a second binary variable, we can look at the `benzene_category` which will be a high vs low binary variable, where the median value is used to determine if an observation is high or low.

In [41]:
median_benzene = air_quality.df.approxQuantile("benzene_gt", [0.5], 0.01)[0]
print(f"The median benzene is: {median_benzene}")

# Create the binary variable
air_quality.df = air_quality.df.withColumn(
    "benzene_category",
    F.when(F.col("benzene_gt").isNull(), None)
     .when(F.col("benzene_gt") <= median_benzene, "Low")
     .otherwise("High")
)

The median benzene is: 8.2


Now we want to provide a few examples (4-5 examples) for each method used in `my_class.py` using the instance of the class from the csv file.

### Method 1: check_numeric_range

This first section of examples will be used to test the `check_numeric_range` method. Here we should expect to see appended Boolean columns when a numeric column is provided. If a non-numeric column is provided, then the dataframe is returned without modification.

#### Example 1

This first example will check if the temperature is within a specific range of two bounds (0 to 12).

In [13]:
air_quality.check_numeric_range("temp", 0, 12).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low|            High|         false|
|3/10/2004|2026-03-26 19:00:00|  1|  2.0|     1292|    112|       9.4|        955|   103|      1174|    92|      1559|      972|13.3|        47.7|      0.7255|          Low|   

26/03/26 21:19:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 2

We now want an example that checks Relative Humidity with only an upper bound of (50).

In [14]:
air_quality.check_numeric_range("rel_humidity", upper = 50).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low|            High|         false|                  true|
|3/10/2004|2026-03-26 19:00:00|  1|  2.0|     1292|    112|       9.4|        955|  

26/03/26 21:21:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 3

Now we want to show an example where we check the benzene levels to have a lower bound of 5.

In [15]:
air_quality.check_numeric_range("benzene_gt", lower = 5).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low|            High|         false|                  true|                true|


26/03/26 21:22:35 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 4

Now we want to try an example where we supply the `check_numeric_range` method a non-numeric column, like the `temp_category` variable.

In [16]:
air_quality.check_numeric_range("temp_category").df.show(10)

Message: Column 'temp_category' is non-numeric.
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low|            High|         fal

26/03/26 21:27:19 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


### Method 2: check_string_levels method

Next we want to test the `check_string_levels` method.

#### Example 1

The first example can be used to check the string levels of the `temp_category` variable. Here we supply both levels to the method.

In [17]:
air_quality.check_string_levels("temp_category", ["High", "Low"]).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   166|      1056|   113|      1692|     1268|13.6|        48.9|      0.7578|          Low

26/03/26 21:35:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 2

Now what if we want to supply just one level of `benzene_category` to the method.

In [18]:
air_quality.check_string_levels("benzene_category", ["Low"]).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+
|3/10/2004|2026-03-26 18:00:00|  0|  2.6|     1360|    150|      11.9|       1046|   

26/03/26 21:37:36 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 3

Now what if we want to see if this method will work with one of our appended Boolean columns.

In [20]:
air_quality.check_string_levels("temp_in_bounds", ["true"]).df.show(10)

Message: Column 'temp_in_bounds' is not a string column.
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+
|3/10/2004|2026-03-26 18:00:

26/03/26 21:39:06 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 4

We know that `Date` is also a string, so let's check which columns are for the date "3/10/2004".

In [21]:
air_quality.check_string_levels("Date", ["3/10/2004"]).df.show(10)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|Date_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+
|3/10/2004|2026-03-26 18:00:00|  0

26/03/26 21:41:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 5

Now we can do a final example with a numeric column to make sure that the message is printed with a numeric column.

In [22]:
air_quality.check_string_levels("co_gt", ["1.2"]).df.show(10)

Message: Column 'co_gt' is not a string column.
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|Date_valid_level|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----

26/03/26 21:43:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


### Method 3: check_missing method

Now we can try a few examples to test the `check_missing` method. Here we only do two, per Dr. Post's Slack message.

#### Example 1

First we want to check for missing in the benzene column.

In [24]:
air_quality.check_missing("benzene_gt").df.show(5)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|Date_valid_level|benzene_gt_is_null|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+--------------

26/03/26 21:48:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


#### Example 2

Now we might want to check missing with a string column.

In [25]:
air_quality.check_missing("temp_category").df.show(5)

+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------+----------------------------+----------------+------------------+---------------------+
|     Date|               Time| id|co_gt|sensor_co|nmhc_gt|benzene_gt|sensor_nmhc|nox_gt|sensor_nox|no2_gt|sensor_no2|sensor_o3|temp|rel_humidity|abs_humidity|temp_category|benzene_category|temp_in_bounds|rel_humidity_in_bounds|benzene_gt_in_bounds|temp_category_valid_level|benzene_category_valid_level|Date_valid_level|benzene_gt_is_null|temp_category_is_null|
+---------+-------------------+---+-----+---------+-------+----------+-----------+------+----------+------+----------+---------+----+------------+------------+-------------+----------------+--------------+----------------------+--------------------+-------------------------

26/03/26 21:49:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/26 21:49:49 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


### Method 4: report_min_max method

Now we can move into our summarization methods and start with the `report_min_max` method that can report the min and max of a user specified variable and has the option to include a grouping variable. These methods will return a pandas data frame.

#### Example 1

The first example allows us to look at the min and max of benzene without any grouping.

In [27]:
air_quality.report_min_max("benzene_gt")

,benzene_gt_min,benzene_gt_max
0,0.1,63.7


#### Example 2

Now we might want to see how these min and max change with a grouping variable, like our `temp_category` variable.

In [28]:
air_quality.report_min_max("benzene_gt", "temp_category")

,temp_category,benzene_gt_min,benzene_gt_max
0,High,0.5,52.1
1,Low,0.1,63.7
2,None,NaN,NaN


#### Example 3

Now we might want to try grouping by a Boolean column, like the appended `temp_in_bounds` column.

In [29]:
air_quality.report_min_max("temp", "temp_in_bounds")

,temp_in_bounds,temp_min,temp_max
0,None,NaN,NaN
1,True,0.0,12.0
2,False,-1.9,44.6


#### Example 4

Now we might want to test what happens if no column is supplied. Here we see that the min and max of all numeric columns are supplied as expected.

In [30]:
air_quality.report_min_max()

26/03/26 21:57:49 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: 
 Schema: _c0
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ambarigh@ncsu.edu/air.csv


,id_min,id_max,co_gt_min,co_gt_max,sensor_co_min,sensor_co_max,nmhc_gt_min,nmhc_gt_max,benzene_gt_min,benzene_gt_max,...,sensor_no2_min,sensor_no2_max,sensor_o3_min,sensor_o3_max,temp_min,temp_max,rel_humidity_min,rel_humidity_max,abs_humidity_min,abs_humidity_max
0,0,9356,0.1,11.9,647,2040,7,1189,0.1,63.7,...,551,2775,221,2523,-1.9,44.6,9.2,88.7,0.1847,2.231


#### Example 5

Finally, we might want to test that a message is printed if a non-numeric column is supplied.

In [31]:
air_quality.report_min_max("temp_category")

Message: Column 'temp_category' is non-numeric.


### Method 5: report_counts method

Now we want to test the `report_counts` method.

#### Example 1

Since we have a few binary variables now, we might want to check the counts of each level for the `temp_category` column.

In [32]:
air_quality.report_counts("temp_category")

,temp_category,count
0,High,4497
1,Low,4494
2,None,366


#### Example 2

Now we want to check this method with two user supplied columns, `temp_category` and `benzene_category`.

In [33]:
air_quality.report_counts("temp_category", "benzene_category")

,temp_category,benzene_category,count
0,None,None,366
1,High,Low,1777
2,Low,Low,2725
3,Low,High,1769
4,High,High,2720


#### Example 3

The method should only allow you to supply to separate arguments, let's test this.

In [34]:
air_quality.report_counts("temp_category", "benzene_category", "Date")

TypeError: report_counts() takes from 2 to 3 positional arguments but 4 were given

#### Example 4

Now we want to see what happens if a numeric column is provided. The instructions did not specify if the summarization was to still be returned.

In [42]:
air_quality.report_counts("benzene_gt")

Message: Column 'benzene_gt' is numeric/non-string.


,benzene_gt,count
0,15.5,27
1,14.9,22
2,13.4,42
3,26.7,6
4,15.4,25
...,...,...
403,39.2,1
404,46.3,1
405,16.4,24
406,29.9,2


### Create an instance from the pandas dataframe

Now that we did a few examples using the instance created from the csv file. We want to read in the same data set using pandas and create an instance of this class from the pandas data frame.

In [43]:
air_raw = pd.read_csv("air.csv")
air_raw.head(5) # Check that it got read in

,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


In [44]:
# Create an instance using our class
air_pdf = SparkDataCheck.from_pandas(spark, air_raw)
air_pdf.df.show(5) # Another quick check to see top 5 rows

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

Again, we need some data cleaning.

In [48]:
# Remap the column names
rename_map = {
    "_c0": "id",
    "CO(GT)": "co_gt",
    "PT08.S1(CO)": "sensor_co",
    "NMHC(GT)": "nmhc_gt",
    "C6H6(GT)": "benzene_gt",
    "PT08.S2(NMHC)": "sensor_nmhc",
    "NOx(GT)": "nox_gt",
    "PT08.S3(NOx)": "sensor_nox",
    "NO2(GT)": "no2_gt",
    "PT08.S4(NO2)": "sensor_no2",
    "PT08.S5(O3)": "sensor_o3",
    "T": "temp",
    "RH": "rel_humidity",
    "AH": "abs_humidity"
}

# Now apply the new names to the old names, and also replace -200 with None
for old, new in rename_map.items():
    if old in air_pdf.df.columns:
        air_pdf.df = air_pdf.df.withColumn(
            new, 
            F.when(F.col(f"`{old}`") == -200, None).otherwise(F.col(f"`{old}`"))
        )
        if old != new:
            air_pdf.df = air_pdf.df.drop(old)
            
# Create temperature binary variable
median_temp = air_pdf.df.approxQuantile("temp", [0.5], 0.01)[0]
print(f"The median temperature is: {median_temp}")

# Create the binary variable
air_pdf.df = air_pdf.df.withColumn(
    "temp_category",
    F.when(F.col("temp").isNull(), None)
     .when(F.col("temp") <= median_temp, "Low")
     .otherwise("High")
)

# Create benzene binary variable
median_benzene = air_pdf.df.approxQuantile("benzene_gt", [0.5], 0.01)[0]
print(f"The median benzene is: {median_benzene}")

# Create the binary variable
air_pdf.df = air_pdf.df.withColumn(
    "benzene_category",
    F.when(F.col("benzene_gt").isNull(), None)
     .when(F.col("benzene_gt") <= median_benzene, "Low")
     .otherwise("High")
)
            
# Now print and check the new schema
print("Cleaned Schema:")
air_pdf.df.printSchema()

The median temperature is: 17.6
The median benzene is: 8.1
Cleaned Schema:
root
 |-- Unnamed: 0: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- co_gt: double (nullable = true)
 |-- sensor_co: long (nullable = true)
 |-- nmhc_gt: long (nullable = true)
 |-- benzene_gt: double (nullable = true)
 |-- sensor_nmhc: long (nullable = true)
 |-- nox_gt: long (nullable = true)
 |-- sensor_nox: long (nullable = true)
 |-- no2_gt: long (nullable = true)
 |-- sensor_no2: long (nullable = true)
 |-- sensor_o3: long (nullable = true)
 |-- temp: double (nullable = true)
 |-- rel_humidity: double (nullable = true)
 |-- abs_humidity: double (nullable = true)
 |-- temp_category: string (nullable = true)
 |-- benzene_category: string (nullable = true)



### Example Using report_min_max with grouping

Now that we have a pandas instance of the air quality data set, we can use the `report_min_max` method to determine the min and max of benzene across a grouping variable (`temp_category`).

In [49]:
air_pdf.report_min_max("benzene_gt", group_var = "temp_category")

,temp_category,benzene_gt_min,benzene_gt_max
0,High,0.5,52.1
1,Low,0.1,63.7
2,None,NaN,NaN


## Part II: NFL analysis using spark

This section of the notebook will focus on practicing some basic analysis using two different Spark API (pandas-on-Spark and Spark SQL) with NFL data.

While both leverage the distributed computing power of the Spark engine, they offer different developer experiences:

**pandas-on-Spark** (`pyspark.pandas`) is designed for data scientists familiar with the pandas API, allowing for a nearly seamless transition from local to distributed environments using similar notation and methods.

**Spark SQL** (`pyspark.sql`) is native to the Spark DataFrame API and uses domain-specific language and functions that are more explicit and optimized for cluster operations.

We will perform identical data engineering tasks such as filtering high-volume players, calculating efficiency metrics like completion percentage and TD/INT ratios, and identifying top performers, to observe how each API handles logic and potential edge cases (like division by zero).

For this section, you will need to make sure you read in the `weekly_nfl_data.csv` into the JupyterHub environment.

### Using **pandas**-on-Spark

This section will focus on using **pandas**-on-Spark to read in the NFL data, check out the rows, look at the QB stats for the 2005 to 2023 seasons, and do some subset and sorting of the rows.

The first step is to read in the NFL data and check out the first five rows with `.head()`.

In [67]:
nfl_psdf = ps.read_csv("weekly_nfl_data.csv", delimiter = ",")
nfl_psdf.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


We then want to report all of the column names.

In [68]:
nfl_psdf.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

Next, we want to do some filtering where we are only looking at the 'QB' stats for the regular season ("REG") within the inclusive 2005 to 2023 seasons. We can view the first five rows to double check that all of the filtering is correct, and see that we start with the 2005 regular season with only QB players.

In [69]:
nfl_filtered_ps = nfl_psdf[
    (nfl_psdf['position'] == "QB") & 
    (nfl_psdf['season_type'] == "REG") & 
    (nfl_psdf['season'] >= 2005) & (nfl_psdf['season'] <= 2023)
]
nfl_filtered_ps.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
29406,00-0000722,None,Tony Banks,QB,QB,None,HOU,2005,17,REG,SF,14,25,173.0,1,2.0,0.0,-0.0,0,0,0.0,0.0,7.0,-3.693779,0,0.0,0.032036,2,-2.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,6.72,6.72
29426,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,9,REG,GB,9,16,65.0,0,1.0,1.0,6.0,1,0,0.0,0.0,4.0,-6.609788,0,0.0,0.025249,8,14.0,0,0.0,0.0,1.0,-1.614163,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,2.00,2.00
29427,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,10,REG,CLE,13,19,150.0,0,0.0,0.0,-0.0,0,0,0.0,0.0,7.0,5.193222,0,0.0,0.171917,3,16.0,1,0.0,0.0,2.0,0.737467,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,13.60,13.60
29428,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,16,REG,CLE,1,1,31.0,1,0.0,0.0,-0.0,0,0,0.0,0.0,1.0,4.662244,0,0.0,NaN,0,0.0,0,0.0,0.0,0.0,NaN,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,5.24,5.24
29447,00-0001335,None,Jeff Blake,QB,QB,None,CHI,2005,2,REG,DET,1,1,11.0,0,0.0,0.0,-0.0,0,0,0.0,0.0,1.0,1.514100,0,0.0,NaN,1,-1.0,0,0.0,0.0,0.0,-2.570310,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,0.34,0.34


We then want to subset the columns to only include the `player_display_name`, `season`, `week`, `completions`,
`attempts`, `passing_yards`, `passing_tds`, and `interceptions`. We again can view the first five rows to check everything looks good, and we do in fact see that there is grouping by `player_display_name` and `season`.

In [70]:
cols_to_keep = ['player_display_name', 'season', 'week', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions']
nfl_filtered_ps = nfl_filtered_ps[cols_to_keep]
nfl_filtered_ps.head()

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


Next we want to look at each `player_display_name` and `season` combination and find the sum and mean of each of the remaining columns from above. Here we can create a separate list of the statistic columns, group the dataframe by the `player_display_name` and `season` and use the `.agg()` method to find the sum and mean.

In [71]:
stats_cols = ['completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions']
nfl_agg_ps = nfl_filtered_ps.groupby(['player_display_name', 'season'])[stats_cols].agg(['sum', 'mean'])
nfl_agg_ps.head()

completions            attempts            passing_yards             passing_tds           interceptions          
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                   
Jake Delhomme       2006           263  20.230769      431  33.153846        2805.0  215.769231          17  1.307692          11.0  0.846154
Jake Plummer        2005           277  17.312500      456  28.500000        3366.0  210.375000          18  1.125000           7.0  0.437500
Matt Schaub         2006            18   3.600000       27   5.400000         208.0   41.600000           1  0.200000           2.0  0.400000
Vince Young         2006           184  12.266667      356  23.733333        2199.0  146.600000          12  0.800000          13.0  0.866667
Kerry Collins       2007            50   8.333333       82  13.666667         531.0   88.500000           0  0.000000           0.0  0.000000

Here we can look at the summarizations created and see that for the first five rows shown that Jake Delhomme's 2006 season has the highest average completions, attempts, passing_yards, passing_tds, and interceptions. However, we see that Jake Plummer's 2005 season had higher sums across most statistic columns. We also can note that compared to the other five players, Matt Schaub had some pretty low statistics in 2006.

Before we create the new variables we might want to flatten the summarization for ease of use. Here we use some list comprehension to combine the original column name c[0] and the aggregation method c[1]

In [72]:
# Before we create the new variables we might want to flatten the summarization for ease of use.
# Here we use some list comprehension to combine the original column name c[0] and the aggregation method c[1]
nfl_agg_ps.columns = [f"{c[0]}_{c[1]}" for c in nfl_agg_ps.columns]
nfl_agg_ps = nfl_agg_ps.reset_index()
nfl_agg_ps.head()

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000


We then want to create two new variables by the season and player combination that focus on the `completion_percentage` and the `td_int_ratio`, both of which can be calculated by the below equations.

`completion_percentage` = (sum of completions) / (sum of attempts)

`td_int_ratio` = (sum passing tds) / (sum interceptions)

In [74]:
nfl_agg_ps['completion_percentage'] = (nfl_agg_ps['completions_sum'] / nfl_agg_ps['attempts_sum']) * 100
nfl_agg_ps['td_int_ratio'] = nfl_agg_ps['passing_tds_sum'] / nfl_agg_ps['interceptions_sum']
nfl_agg_ps.head()

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,61.020882,1.545455
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,60.745614,2.571429
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000,66.666667,0.500000
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,51.685393,0.923077
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,60.975610,NaN


Upon looking at the newly created variables among the first five players, maybe Matt Schaub isn't so bad as he has the highest `completion_percentage` with a 66.67%. But he also has the lowest touchdown and interception ratio at 0.5. Also out of the five, Jake Plummer has a pretty good touchdown and interception ratio at 2.57 and a fairly decent completion percentage at 60.75%.

We will then save the result above as an object, and subset the rows to only include player/season combinations where the sum of attempts was at least 50. Here we see that our poor Matt Schaub got booted off the list.

In [76]:
nfl_final_ps = nfl_agg_ps[nfl_agg_ps['attempts_sum'] >= 50]
nfl_final_ps.head()

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,61.020882,1.545455
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,60.745614,2.571429
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,51.685393,0.923077
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,60.975610,NaN
6,Charlie Batch,2006,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,57.692308,inf


We then want to sort the rows in descending order by `completion_percentage` and report the first 40 values.

Here we see that C.J. Beathard has the highest completion_percentage at 75.47%, although there are quite a few that follow closely behind him.

In [78]:
print("\nTop 40 by Completion Percentage (pandas-on-Spark):")
print(nfl_final_ps.sort_values(by="completion_percentage", ascending=False).head(40))


Top 40 by Completion Percentage (pandas-on-Spark):
     player_display_name  season  completions_sum  completions_mean  attempts_sum  attempts_mean  passing_yards_sum  passing_yards_mean  passing_tds_sum  passing_tds_mean  interceptions_sum  interceptions_mean  completion_percentage  td_int_ratio
1409       C.J. Beathard    2023               40          6.666667            53       8.833333              349.0           58.166667                1          0.166667                0.0            0.000000              75.471698           inf
1248          Colt McCoy    2021               74         10.571429            99      14.142857              740.0          105.714286                3          0.428571                1.0            0.142857              74.747475      3.000000
900          Matt Schaub    2019               50         10.000000            67      13.400000              580.0          116.000000                3          0.600000                1.0            0.2000

Next, we want to sort the rows in descending order by the `td_int_ratio` and report the first 40 values. Something we do want to point out is that some of the players appear to have `inf` for their `td_int_ratio`. This is due to the **pandas**-on-Spark API treating a divide by zero `interceptions_sum` to result in `inf`. And any players whose `td_int_ratio` is `inf` appears at the top of the list with descending order. After passing over these players, we see that the famous Tom Brady has the highest `td_int_ratio` of this subset.

In [79]:
print("\nTop 40 by TD/INT Ratio (pandas-on-Spark):")
print(nfl_final_ps.sort_values(by="td_int_ratio", ascending=False).head(40))


Top 40 by TD/INT Ratio (pandas-on-Spark):
     player_display_name  season  completions_sum  completions_mean  attempts_sum  attempts_mean  passing_yards_sum  passing_yards_mean  passing_tds_sum  passing_tds_mean  interceptions_sum  interceptions_mean  completion_percentage  td_int_ratio
6          Charlie Batch    2006               30          4.285714            52       7.428571              477.0           68.142857                5          0.714286                0.0            0.000000              57.692308           inf
26           Matt Schaub    2005               33          4.714286            64       9.142857              495.0           70.714286                4          0.571429                0.0            0.000000              51.562500           inf
73          Todd Collins    2007               67         16.750000           105      26.250000              888.0          222.000000                5          1.250000                0.0            0.000000       